# Benchmark question embeddings for both retrieval models

Embeds the question of every target benchmark the pipeline supports (**numina_hard** = the NuminaMath-1.5 hard set,
**math500**, **gsm8k**, **aime25**, **aime26**) with both embedding models, exactly as retrieval embeds them, and uploads
them to `mostafabehroozi/numinamath-2025`:

```
benchmark_embeddings/Bert-MLM_arXiv-MP-class_zbMath/   manifest.json, <benchmark>.npy, <benchmark>_texts.json
benchmark_embeddings/MathLeap-Qwen-8B/                 the same files
```

With `CONFIG["BENCHMARK_EMBEDDINGS_ENABLED"] = True` the pipeline reads these vectors (`src/query_embeddings.py`) and
never loads the model for these questions, so MathLeap-Qwen-8B runs on a CPU notebook (it takes ~77 s per question there).

* The question texts come from the repository's own loaders (`src/benchmark_data.py`), so they are the exact strings
  retrieval embeds; vectors are matched by that text.
* **BERT**: `SentenceTransformer(...).encode(question)`, as `src/utils.load_embedding_model` does.
* **MathLeap-Qwen-8B**: the recipe of its corpus file (pinned revision, 512 tokens, no prompt, last-token pooling,
  L2-normalised), run in float16 on one T4 with the token-embedding table in CPU memory.
* **Check**: numina_hard questions are corpus rows, so their vectors must equal the rows of the published corpus
  embedding files; the run stops if they do not.

**Before running**: *GPU T4 x2*, *Internet on*. With a Hugging Face write token (`HF_TOKEN_DIRECT` in the first cell, or a Kaggle secret `HF_TOKEN`) the folders are uploaded;
without it they are saved as notebook output (`/kaggle/working/benchmark_embeddings`). About 15 minutes.

In [ ]:
# Settings
HF_TOKEN_DIRECT = ""  # paste a Hugging Face write token between the quotes to upload from here (or use the Kaggle secret HF_TOKEN)
REPO_URL = "https://github.com/mostafabehroozi/analogical_math_rag.git"
REPO_BRANCH = "analogical-consistency"
TARGET_REPO = "mostafabehroozi/numinamath-2025"
FOLDER_IN_REPO = "benchmark_embeddings"
BENCHMARKS = ["numina_hard", "math500", "gsm8k", "aime25", "aime26"]
HARD_FILE = ("hard_questions/hard_questions_n2000_farthest_src-amc_aime20-cn_contest10-aops_forum10-olympiads_ref7-"
             "number_theory3-cn_k1235-other15_seed42_20261119_023944.json")
MODELS = ["Bert-MLM_arXiv-MP-class_zbMath", "MathLeap-Qwen-8B"]
CHECK_ROWS = 64          # numina_hard rows compared with the corpus embedding files
TOKENS_PER_BATCH = 8192  # MathLeap padded tokens per forward pass
WORK = "/tmp/benchmark_embeddings"

In [ ]:
# Token, repository code, and the target repo
import datetime, json, os, shutil, subprocess, sys, time
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = HF_TOKEN_DIRECT.strip() or None
if not HF_TOKEN:
    try:  # optional: without a token the files are only saved to /kaggle/working/benchmark_embeddings
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
print("Hugging Face token:", "yes, files will be uploaded" if HF_TOKEN else "none, files are saved as notebook output")
api.repo_info(TARGET_REPO, repo_type="dataset")

CODE_DIR = "/tmp/analogical_math_rag"
shutil.rmtree(CODE_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, CODE_DIR], check=True)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
from src.corpus_filter import EXEMPLAR_CORPUS_PRESETS, EMBEDDING_MODEL_PRESETS, embedding_preset
from src.query_embeddings import benchmark_question_texts, write_model_folder, text_key
from src.benchmark_data import HUGGINGFACE_BENCHMARK_SPECS
print("code commit:", CODE_COMMIT)
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
STARTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()

In [ ]:
# The question texts, built by the pipeline's own loaders
import numpy as np
from datasets import load_dataset

CORPUS = EXEMPLAR_CORPUS_PRESETS["NuminaMath-1.5"]
t0 = time.time()
corpus_questions = load_dataset(CORPUS["hf_name"], split="train", revision=CORPUS["revision"])["problem"]
hard_indices = json.load(open(hf_hub_download(TARGET_REPO, HARD_FILE, repo_type="dataset", token=HF_TOKEN)))
TEXTS = benchmark_question_texts(BENCHMARKS, hard_indices=hard_indices, corpus_questions=corpus_questions)
del corpus_questions

SOURCES = {"numina_hard": {"source": f"{CORPUS['hf_name']}@{CORPUS['revision']} rows of {TARGET_REPO}/{HARD_FILE}",
                           "corpus_rows": hard_indices}}
for name in BENCHMARKS:
    if name in HUGGINGFACE_BENCHMARK_SPECS:
        spec = HUGGINGFACE_BENCHMARK_SPECS[name]
        SOURCES[name] = {"source": f"{spec['path']} ({spec.get('name', 'default')}, {spec['split']})",
                         "dataset_revision": api.dataset_info(spec["path"]).sha}
for name, texts in TEXTS.items():
    print(f"{name}: {len(texts)} questions, {len(set(texts))} distinct")
print(f"loaded in {time.time() - t0:.0f}s")

In [ ]:
# Reading a few rows of a published corpus embedding file (downloaded once to /tmp, pinned, then memory-mapped)
def corpus_rows(model_name, rows):
    files = embedding_preset({"EXEMPLAR_CORPUS": "NuminaMath-1.5", "EXEMPLAR_EMBEDDING_MODEL": model_name})
    src = files["embedding_source"]
    path = hf_hub_download(src["repo_id"], src["filename"], repo_type=src["repo_type"], revision=src["revision"],
                           local_dir="/tmp/corpus_embeddings", token=HF_TOKEN)
    assert os.path.getsize(path) == src["bytes"], "corpus embedding file size differs"
    out = np.asarray(np.load(path, mmap_mode="r")[list(rows)], dtype=np.float32)
    os.remove(path)  # 3-7 GB, only needed for this check
    return out


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)


CHECK_POSITIONS = np.random.default_rng(0).choice(len(hard_indices), size=CHECK_ROWS, replace=False)
VECTORS, CHECKS, MODEL_INFO = {}, {}, {}

In [ ]:
# BERT: SentenceTransformer.encode, as the pipeline does
import torch
from sentence_transformers import SentenceTransformer
import sentence_transformers, transformers

name = "Bert-MLM_arXiv-MP-class_zbMath"
preset = EMBEDDING_MODEL_PRESETS[name]
t0 = time.time()
bert = SentenceTransformer(preset["hf_name"], device="cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
VECTORS[name] = {b: bert.encode(texts, batch_size=64, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
                 for b, texts in TEXTS.items()}
ref = corpus_rows(name, [hard_indices[p] for p in CHECK_POSITIONS])
cos = (unit(VECTORS[name]["numina_hard"][CHECK_POSITIONS]) * unit(ref)).sum(1)
CHECKS[name] = {"corpus_rows_checked": CHECK_ROWS, "cosine_min": float(cos.min()), "cosine_mean": float(cos.mean())}
MODEL_INFO[name] = {"model_id": preset["hf_name"], "model_revision": api.model_info(preset["hf_name"]).sha,
                    "max_seq_length": bert.max_seq_length, "normalized": False,
                    "recipe": f"SentenceTransformer('{preset['hf_name']}').encode(question)"}
print(name, {b: v.shape for b, v in VECTORS[name].items()}, CHECKS[name], f"{time.time() - t0:.0f}s")
assert cos.min() > 0.999, "BERT vectors differ from the corpus file rows of the same questions"
del bert
torch.cuda.empty_cache()

In [ ]:
# MathLeap-Qwen-8B: the corpus file's recipe in float16 on one T4 (token-embedding table stays in CPU memory)
import gc
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from pathlib import Path
from transformers import AutoModel, AutoTokenizer
from packaging.version import Version

name = "MathLeap-Qwen-8B"
preset = EMBEDDING_MODEL_PRESETS[name]
MAX_SEQ = preset["max_seq_length"]
t0 = time.time()
path = Path(snapshot_download(preset["hf_name"], revision=preset["revision"], token=HF_TOKEN))
pooling = json.loads((path / "1_Pooling" / "config.json").read_text())
assert pooling.get("pooling_mode_lasttoken")
tok = AutoTokenizer.from_pretrained(path)
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
LEFT = tok.padding_side == "left"
DTYPE_KW = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
dev = "cuda:0"
m = AutoModel.from_pretrained(path, **{DTYPE_KW: torch.float16}, attn_implementation="sdpa").eval()
emb = m.get_input_embeddings()
for child_name, child in m.named_children():
    if child is not emb:
        child.to(dev)
for p in list(m.parameters(recurse=False)) + list(m.buffers(recurse=False)):
    p.data = p.data.to(dev)
gc.collect()
print(f"loaded in {time.time() - t0:.0f}s")


@torch.inference_mode()
def forward(ids_list):
    n, L = len(ids_list), max(len(x) for x in ids_list)
    ids = torch.full((n, L), PAD, dtype=torch.long)
    mask = torch.zeros((n, L), dtype=torch.long)
    for r, x in enumerate(ids_list):
        sl = slice(L - len(x), L) if LEFT else slice(0, len(x))
        ids[r, sl] = torch.tensor(x)
        mask[r, sl] = 1
    h = m(inputs_embeds=F.embedding(ids, emb.weight).to(dev), attention_mask=mask.to(dev), use_cache=False).last_hidden_state
    last = torch.full((n,), L - 1) if LEFT else mask.sum(1) - 1
    return F.normalize(h[torch.arange(n, device=dev), last.to(dev)].float(), p=2, dim=1).cpu().numpy()


def encode(texts):
    # SentenceTransformer strips each text; the corpus file used no prompt and 512 tokens.
    ids = tok([t.strip() for t in texts], truncation=True, max_length=MAX_SEQ)["input_ids"]
    order = np.argsort([-len(x) for x in ids], kind="stable")
    out = np.zeros((len(ids), m.config.hidden_size), np.float32)
    i = 0
    while i < len(order):
        width = len(ids[order[i]])
        size = max(1, min(64, TOKENS_PER_BATCH // width))
        batch = order[i:i + size]
        out[batch] = forward([ids[j] for j in batch])
        i += size
    return out


t0 = time.time()
VECTORS[name] = {b: encode(texts) for b, texts in TEXTS.items()}
ref = corpus_rows(name, [hard_indices[p] for p in CHECK_POSITIONS])
cos = (unit(VECTORS[name]["numina_hard"][CHECK_POSITIONS]) * unit(ref)).sum(1)
CHECKS[name] = {"corpus_rows_checked": CHECK_ROWS, "cosine_min": float(cos.min()), "cosine_mean": float(cos.mean())}
MODEL_INFO[name] = {"model_id": preset["hf_name"], "model_revision": preset["revision"], "max_seq_length": MAX_SEQ,
                    "normalized": True, "prompt": "", "compute_dtype": "float16",
                    "recipe": f"m = SentenceTransformer('{preset['hf_name']}', revision='{preset['revision']}'); "
                              f"m.max_seq_length = {MAX_SEQ}; m.encode(question)  (no prompt, last token, L2-normalised)"}
print(name, {b: v.shape for b, v in VECTORS[name].items()}, CHECKS[name], f"{time.time() - t0:.0f}s")
assert cos.min() > 0.99, "MathLeap vectors differ from the corpus file rows of the same questions"
del m
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Write both folders (saved as notebook output) and upload them when a token is set
OUT = os.path.join("/kaggle/working", FOLDER_IN_REPO)
shutil.rmtree(OUT, ignore_errors=True)
MANIFESTS = {}
for name in MODELS:
    info = {**MODEL_INFO[name], "created_at": STARTED_AT, "code_commit": CODE_COMMIT,
            "torch": torch.__version__, "transformers": transformers.__version__,
            "sentence_transformers": sentence_transformers.__version__, "check_against_corpus_file": CHECKS[name]}
    MANIFESTS[name] = write_model_folder(os.path.join(OUT, name), info, TEXTS, VECTORS[name], SOURCES)
    print(name, json.dumps({k: v for k, v in MANIFESTS[name].items() if k != "benchmarks"}, indent=1)[:1500])
UPLOAD_REVISION = None
if HF_TOKEN:
    commit = api.upload_folder(folder_path=OUT, path_in_repo=FOLDER_IN_REPO, repo_id=TARGET_REPO, repo_type="dataset",
                               commit_message=f"Benchmark question embeddings ({', '.join(BENCHMARKS)}) for {', '.join(MODELS)}")
    UPLOAD_REVISION = commit.oid
    print("uploaded, revision:", UPLOAD_REVISION)
else:
    print("no token: upload", OUT, f"to {TARGET_REPO}/{FOLDER_IN_REPO} from elsewhere")

In [ ]:
# Read them back through the pipeline's own loader (from Hugging Face when uploaded): every question must be found
from src.query_embeddings import ensure_benchmark_embeddings, load_saved_vectors
for name in MODELS:
    loaded = load_saved_vectors(os.path.join(OUT, name), {"EXEMPLAR_CORPUS": "NuminaMath-1.5", "EXEMPLAR_EMBEDDING_MODEL": name})
    assert all(text_key(t) in loaded["vectors"] for texts in TEXTS.values() for t in texts)
    print("local folder OK:", name, loaded["counts"])
from src.query_embeddings import ensure_benchmark_embeddings, load_saved_vectors
for name in (MODELS if UPLOAD_REVISION else []):
    source = {"repo_id": TARGET_REPO, "repo_type": "dataset", "revision": UPLOAD_REVISION,
              "folder": f"{FOLDER_IN_REPO}/{name}"}
    EMBEDDING_MODEL_PRESETS[name]["benchmark_embeddings_source"] = source
    config = {"EXEMPLAR_CORPUS": "NuminaMath-1.5", "EXEMPLAR_EMBEDDING_MODEL": name,
              "EMBEDDINGS_DIR": os.path.join(WORK, "readback")}
    loaded = load_saved_vectors(ensure_benchmark_embeddings(config), config)
    found = sum(text_key(t) in loaded["vectors"] for texts in TEXTS.values() for t in texts)
    total = sum(len(t) for t in TEXTS.values())
    print(name, loaded["counts"], f"found {found}/{total}")
    assert found == total
print(json.dumps({"revision": UPLOAD_REVISION, "checks": CHECKS,
                  "sources": {n: {k: v for k, v in s.items() if k != "corpus_rows"} for n, s in SOURCES.items()}}, indent=2))